In [9]:
  from pynq import Overlay
  import numpy as np

  # Bitstream을 FPGA에 다운로드
  overlay = Overlay('mobilenet.bit')
  print("Overlay loaded")

  # 디자인에 들어있는 IP 목록
  print("\nIP blocks:")
  for name in overlay.ip_dict.keys():
      print(f"  - {name}")


Overlay loaded

IP blocks:
  - MobileNet_Stream_0
  - axi_gpio_0
  - axi_dma_0
  - axi_dma_1
  - axi_dma_2
  - processing_system7_0


In [10]:
  from pynq import Overlay, allocate
  import numpy as np

  overlay = Overlay('mobilenet.bit')

  # HLS IP의 컨트롤 인터페이스 자세히 보기
  mn = overlay.MobileNet_Stream_0
  print("=== MobileNet_Stream_0 register map ===")
  print(mn.register_map)
  print()
  print("=== Signature ===")
  print(mn.signature)



=== MobileNet_Stream_0 register map ===
RegisterMap {
  ext_w_conv_1 = Register(ext_w_conv=write-only),
  ext_w_conv_2 = Register(ext_w_conv=write-only),
  ext_b_conv_1 = Register(ext_b_conv=write-only),
  ext_b_conv_2 = Register(ext_b_conv=write-only),
  ext_w_fc_1 = Register(ext_w_fc=write-only),
  ext_w_fc_2 = Register(ext_w_fc=write-only),
  ext_b_fc_1 = Register(ext_b_fc=write-only),
  ext_b_fc_2 = Register(ext_b_fc=write-only),
  ext_tile_1 = Register(ext_tile=write-only),
  ext_tile_2 = Register(ext_tile=write-only),
  ext_info_1 = Register(ext_info=write-only),
  ext_info_2 = Register(ext_info=write-only)
}

=== Signature ===
None


In [11]:
  for name in ['axi_dma_0', 'axi_dma_1', 'axi_dma_2']:
      dma = getattr(overlay, name)
      print(f"\n=== {name} ===")
      print(f"  sendchannel:    {hasattr(dma, 'sendchannel')}")
      print(f"  recvchannel:    {hasattr(dma, 'recvchannel')}")
      if hasattr(dma, 'sendchannel'):
          print(f"  send max bytes: {dma.sendchannel._max_size}")
      if hasattr(dma, 'recvchannel'):
          print(f"  recv max bytes: {dma.recvchannel._max_size}")



=== axi_dma_0 ===
  sendchannel:    True
  recvchannel:    True
  send max bytes: 8388607


AttributeError: 'NoneType' object has no attribute '_max_size'

In [12]:
  from pynq import Overlay
  overlay = Overlay('mobilenet.bit')

  # 1. ip_dict 전체 자세히 보기
  print("=" * 60)
  print("=== ALL IP entries with full info ===")
  print("=" * 60)
  for name, info in overlay.ip_dict.items():
      print(f"\n[{name}]")
      print(f"  type:    {info.get('type', 'N/A')}")
      print(f"  addr:    0x{info.get('phys_addr', 0):08X}")
      print(f"  range:   0x{info.get('addr_range', 0):08X}")
      print(f"  driver:  {info.get('driver', 'N/A')}")

  # 2. MobileNet_Stream_0 메모리 직접 읽기 (control register 후보)
  print("\n" + "=" * 60)
  print("=== Raw register dump (offset 0x00 ~ 0x40) ===")
  print("=" * 60)
  mn = overlay.MobileNet_Stream_0
  for offset in range(0x00, 0x40, 4):
      try:
          val = mn.read(offset)
          print(f"  0x{offset:02X}: 0x{val:08X}")
      except Exception as e:
          print(f"  0x{offset:02X}: ERROR ({e})")

  # 3. DMA 상태 정확히 확인
  print("\n" + "=" * 60)
  print("=== DMA status (with None check) ===")
  print("=" * 60)
  for name in ['axi_dma_0', 'axi_dma_1', 'axi_dma_2']:
      dma = getattr(overlay, name)
      print(f"\n[{name}]")
      sc = dma.sendchannel
      rc = dma.recvchannel
      print(f"  sendchannel (MM2S): {'ENABLED, max=' + str(sc._max_size) + ' bytes' if sc is not None else 'DISABLED'}")
      print(f"  recvchannel (S2MM): {'ENABLED, max=' + str(rc._max_size) + ' bytes' if rc is not None else 'DISABLED'}")


=== ALL IP entries with full info ===

[MobileNet_Stream_0]
  type:    xilinx.com:hls:MobileNet_Stream:1.0
  addr:    0x40010000
  range:   0x00010000
  driver:  <class 'pynq.overlay.DefaultIP'>

[axi_gpio_0]
  type:    xilinx.com:ip:axi_gpio:2.0
  addr:    0x40020000
  range:   0x00010000
  driver:  <class 'pynq.lib.axigpio.AxiGPIO'>

[axi_dma_0]
  type:    xilinx.com:ip:axi_dma:7.1
  addr:    0x41E00000
  range:   0x00010000
  driver:  <class 'pynq.lib.dma.DMA'>

[axi_dma_1]
  type:    xilinx.com:ip:axi_dma:7.1
  addr:    0x41E10000
  range:   0x00010000
  driver:  <class 'pynq.lib.dma.DMA'>

[axi_dma_2]
  type:    xilinx.com:ip:axi_dma:7.1
  addr:    0x41E20000
  range:   0x00010000
  driver:  <class 'pynq.lib.dma.DMA'>

[processing_system7_0]
  type:    xilinx.com:ip:processing_system7:5.5
  addr:    0x00000000
  range:   0x00000000
  driver:  <class 'pynq.overlay.DefaultIP'>

=== Raw register dump (offset 0x00 ~ 0x40) ===
  0x00: 0x00000000
  0x04: 0x00000000
  0x08: 0x00000000
  

In [13]:
  import xml.etree.ElementTree as ET

  # Jupyter에서 hwh 파일 직접 파싱
  tree = ET.parse('mobilenet.hwh')
  root = tree.getroot()

  # MobileNet_Stream과 관련된 인터페이스 찾기
  print("=== MobileNet_Stream 관련 인터페이스 ===")
  for module in root.iter('MODULE'):
      inst = module.get('INSTANCE', '')
      if 'MobileNet' in inst:
          print(f"\n[{inst}]")
          for port in module.iter('PORT'):
              name = port.get('NAME', '')
              if 's_axi' in name.lower() or 'ctrl' in name.lower() or 'control' in name.lower():
                  print(f"  Port: {name}")

  print("\n=== 모든 메모리 매핑 ===")
  for mmap in root.iter('MEMRANGE'):
      instance = mmap.get('INSTANCE', '')
      if 'MobileNet' in instance:
          base = mmap.get('BASEVALUE', '')
          high = mmap.get('HIGHVALUE', '')
          print(f"  {instance}: {base} ~ {high}")


=== MobileNet_Stream 관련 인터페이스 ===

[MobileNet_Stream_0]
  Port: s_axi_CTRL_BUS_ARADDR
  Port: s_axi_CTRL_BUS_ARREADY
  Port: s_axi_CTRL_BUS_ARVALID
  Port: s_axi_CTRL_BUS_AWADDR
  Port: s_axi_CTRL_BUS_AWREADY
  Port: s_axi_CTRL_BUS_AWVALID
  Port: s_axi_CTRL_BUS_BREADY
  Port: s_axi_CTRL_BUS_BRESP
  Port: s_axi_CTRL_BUS_BVALID
  Port: s_axi_CTRL_BUS_RDATA
  Port: s_axi_CTRL_BUS_RREADY
  Port: s_axi_CTRL_BUS_RRESP
  Port: s_axi_CTRL_BUS_RVALID
  Port: s_axi_CTRL_BUS_WDATA
  Port: s_axi_CTRL_BUS_WREADY
  Port: s_axi_CTRL_BUS_WSTRB
  Port: s_axi_CTRL_BUS_WVALID
  Port: s_axi_control_ARADDR
  Port: s_axi_control_ARREADY
  Port: s_axi_control_ARVALID
  Port: s_axi_control_AWADDR
  Port: s_axi_control_AWREADY
  Port: s_axi_control_AWVALID
  Port: s_axi_control_BREADY
  Port: s_axi_control_BRESP
  Port: s_axi_control_BVALID
  Port: s_axi_control_RDATA
  Port: s_axi_control_RREADY
  Port: s_axi_control_RRESP
  Port: s_axi_control_RVALID
  Port: s_axi_control_WDATA
  Port: s_axi_control_WREADY


In [14]:
  from pynq import MMIO

  # CTRL_BUS는 0x40000000에 매핑됨 (PYNQ가 자동 인식 못함)
  ctrl = MMIO(0x40000000, 0x10000)

  print("=== CTRL_BUS dump (0x40000000) ===")
  for offset in [0x00, 0x04, 0x08, 0x0C, 0x10, 0x14, 0x18, 0x1C, 0x20]:
      val = ctrl.read(offset)
      print(f"  0x{offset:02X}: 0x{val:08X}")

  # AP control 비트 해석
  ap_ctrl = ctrl.read(0x00)
  print(f"\n=== AP_CTRL 해석 (0x{ap_ctrl:08X}) ===")
  print(f"  ap_start:  {(ap_ctrl >> 0) & 1}")
  print(f"  ap_done:   {(ap_ctrl >> 1) & 1}")
  print(f"  ap_idle:   {(ap_ctrl >> 2) & 1}")
  print(f"  ap_ready:  {(ap_ctrl >> 3) & 1}")


=== CTRL_BUS dump (0x40000000) ===
  0x00: 0x00000004
  0x04: 0x00000000
  0x08: 0x00000000
  0x0C: 0x00000000
  0x10: 0x00000000
  0x14: 0x00000000
  0x18: 0x00000000
  0x1C: 0x00000000
  0x20: 0x00000000

=== AP_CTRL 해석 (0x00000004) ===
  ap_start:  0
  ap_done:   0
  ap_idle:   1
  ap_ready:  0


In [15]:
  # layer 레지스터에 값 써보고 다시 읽기 (통신 검증)
  ctrl.write(0x10, 0x12345678)
  val_back = ctrl.read(0x10)
  print(f"layer 레지스터 W/R 검증:")
  print(f"  Wrote: 0x12345678")
  print(f"  Read:  0x{val_back:08X}")
  print(f"  Match: {'✅' if val_back == 0x12345678 else '❌'}")

  # 원복
  ctrl.write(0x10, 0)


layer 레지스터 W/R 검증:
  Wrote: 0x12345678
  Read:  0x12345678
  Match: ✅


In [16]:

  import time

  class MobileNetIP:
      def __init__(self, base_ctrl=0x40000000, base_data=0x40010000):
          self.ctrl = MMIO(base_ctrl, 0x10000)   # ap_ctrl, layer, ...
          self.data = MMIO(base_data, 0x10000)   # 포인터들

      # --- AP control ---
      def is_idle(self):
          return (self.ctrl.read(0x00) >> 2) & 1

      def is_done(self):
          return (self.ctrl.read(0x00) >> 1) & 1

      def start(self):
          self.ctrl.write(0x00, 1)  # ap_start = 1

      def wait(self, timeout=10.0):
          t0 = time.time()
          while not self.is_done():
              if time.time() - t0 > timeout:
                  raise TimeoutError(f"IP did not finish in {timeout}s")
              time.sleep(0.001)

      # --- Scalar inputs ---
      def set_layer(self, layer, inter_layer, type_layer):
          self.ctrl.write(0x10, layer)
          self.ctrl.write(0x18, inter_layer)
          self.ctrl.write(0x20, type_layer)

      # --- Pointer inputs (64-bit) ---
      def _set_ptr(self, offset, addr):
          self.data.write(offset,     addr & 0xFFFFFFFF)
          self.data.write(offset + 4, (addr >> 32) & 0xFFFFFFFF)

      def set_pointers(self, w_conv_addr, b_conv_addr, w_fc_addr, b_fc_addr,
                       tile_addr, info_addr):
          self._set_ptr(0x10, w_conv_addr)
          self._set_ptr(0x1C, b_conv_addr)
          self._set_ptr(0x28, w_fc_addr)
          self._set_ptr(0x34, b_fc_addr)
          self._set_ptr(0x40, tile_addr)
          self._set_ptr(0x4C, info_addr)

  # 인스턴스 생성 후 검증
  mn = MobileNetIP()
  print(f"IP idle: {mn.is_idle()}")
  print(f"IP done: {mn.is_done()}")


IP idle: 1
IP done: 0


In [1]:
# 1. 데이터 파일 파싱

import numpy as np
import time

W_CONV_LAYER = 1525656
B_CONV_LAYER = 17056
FC_LAYER = 360000
B_FC_LAYER = 1000
IMAGE_SIZE = 224 * 224 * 3

def load_dat_text(filepath):
    t0 = time.time()
    with open(filepath, 'r') as f:
        text = f.read()
    arr = np.fromstring(text, sep=' ', dtype=np.int32)
    print(f"  {filepath}: {len(arr):,} ints in {time.time()-t0:.2f}s")
    return arr

print("=== 1. Image ===")
image_data = load_dat_text('image_int.dat')[:IMAGE_SIZE]
print(f"  shape: {image_data.shape}, range: [{image_data.min()}, {image_data.max()}]")

print("\n=== 2. Weights ===")
all_w = load_dat_text('weights.dat')
weights_CONV = all_w[:W_CONV_LAYER]
weights_FC = all_w[W_CONV_LAYER:W_CONV_LAYER + FC_LAYER]
print(f"  weights_CONV: {len(weights_CONV):,}, range: [{weights_CONV.min()}, {weights_CONV.max()}]")
print(f"  weights_FC:   {len(weights_FC):,}, range: [{weights_FC.min()}, {weights_FC.max()}]")

print("\n=== 3. Bias ===")
all_b = load_dat_text('bias.dat')
bias_CONV = all_b[:B_CONV_LAYER]
bias_FC = all_b[B_CONV_LAYER:B_CONV_LAYER + B_FC_LAYER]
print(f"  bias_CONV: {len(bias_CONV):,}, range: [{bias_CONV.min()}, {bias_CONV.max()}]")
print(f"  bias_FC:   {len(bias_FC):,}, range: [{bias_FC.min()}, {bias_FC.max()}]")

print("\nDONE")
print("test")


=== 1. Image ===
  image_int.dat: 150,528 ints in 0.11s
  shape: (150528,), range: [-247, 337]

=== 2. Weights ===
  weights.dat: 1,885,656 ints in 1.39s
  weights_CONV: 1,525,656, range: [-4288, 36727]
  weights_FC:   360,000, range: [0, 1400]

=== 3. Bias ===
  bias.dat: 57,056 ints in 0.04s
  bias_CONV: 17,056, range: [-2278400, 2677760]
  bias_FC:   1,000, range: [-48, 48]

DONE
test


In [2]:
  import os
  import numpy as np
  os.chdir('/home/xilinx/jupyter_notebooks/mobilenet')

  bins = ['tile_3x3','tile_convs','tile_avg','tile_fc',
          'info_3x3','info_convs','info_avg','info_fc']
  expected = [768, 64800, 120, 1920, 4864, 410400, 760, 12160]

  for name, n in zip(bins, expected):
      a = np.fromfile(f'{name}.bin', dtype=np.int32)
      ok = '✓' if len(a) == n else '✗'
      print(f"  {ok} {name}: {len(a):,} ints (expected {n:,})")


  ✓ tile_3x3: 768 ints (expected 768)
  ✓ tile_convs: 64,800 ints (expected 64,800)
  ✓ tile_avg: 120 ints (expected 120)
  ✓ tile_fc: 1,920 ints (expected 1,920)
  ✓ info_3x3: 4,864 ints (expected 4,864)
  ✓ info_convs: 410,400 ints (expected 410,400)
  ✓ info_avg: 760 ints (expected 760)
  ✓ info_fc: 12,160 ints (expected 12,160)


In [3]:
  import os, numpy as np, time
  os.chdir('/home/xilinx/jupyter_notebooks/mobilenet')
# 1) 데이터 로드
  W_CONV_LAYER = 1525656
  B_CONV_LAYER = 17056
  FC_LAYER     = 360000
  B_FC_LAYER   = 1000
  IMAGE_SIZE   = 224*224*3

  def load_dat(fp):
      t0 = time.time()
      with open(fp,'r') as f: a = np.fromstring(f.read(), sep=' ',
  dtype=np.int32)
      print(f"  {fp}: {len(a):,} ints in {time.time()-t0:.1f}s")
      return a

  image_data   = load_dat('image_int.dat')[:IMAGE_SIZE]
  all_w        = load_dat('weights.dat')
  weights_CONV = all_w[:W_CONV_LAYER]
  weights_FC   = all_w[W_CONV_LAYER:W_CONV_LAYER+FC_LAYER]
  all_b        = load_dat('bias.dat')
  bias_CONV    = all_b[:B_CONV_LAYER]
  bias_FC      = all_b[B_CONV_LAYER:B_CONV_LAYER+B_FC_LAYER]

  tile_3x3   = np.fromfile('tile_3x3.bin',   dtype=np.int32)
  tile_convs = np.fromfile('tile_convs.bin', dtype=np.int32)
  tile_avg   = np.fromfile('tile_avg.bin',   dtype=np.int32)
  tile_fc    = np.fromfile('tile_fc.bin',    dtype=np.int32)
  info_3x3   = np.fromfile('info_3x3.bin',   dtype=np.int32)
  info_convs = np.fromfile('info_convs.bin', dtype=np.int32)
  info_avg   = np.fromfile('info_avg.bin',   dtype=np.int32)
  info_fc    = np.fromfile('info_fc.bin',    dtype=np.int32)

  print()
  print(f"image      : {len(image_data):>10,} ints")
  print(f"w_conv     : {len(weights_CONV):>10,} ints")
  print(f"w_fc       : {len(weights_FC):>10,} ints")
  print(f"b_conv     : {len(bias_CONV):>10,} ints")
  print(f"b_fc       : {len(bias_FC):>10,} ints")
  print(f"tile_3x3   : {len(tile_3x3):>10,} ints")
  print(f"tile_convs : {len(tile_convs):>10,} ints")
  print(f"tile_avg   : {len(tile_avg):>10,} ints")
  print(f"tile_fc    : {len(tile_fc):>10,} ints")
  print(f"info_3x3   : {len(info_3x3):>10,} ints")
  print(f"info_convs : {len(info_convs):>10,} ints")
  print(f"info_avg   : {len(info_avg):>10,} ints")
  print(f"info_fc    : {len(info_fc):>10,} ints")


  image_int.dat: 150,528 ints in 0.1s
  weights.dat: 1,885,656 ints in 1.0s
  bias.dat: 57,056 ints in 0.0s

image      :    150,528 ints
w_conv     :  1,525,656 ints
w_fc       :    360,000 ints
b_conv     :     17,056 ints
b_fc       :      1,000 ints
tile_3x3   :        768 ints
tile_convs :     64,800 ints
tile_avg   :        120 ints
tile_fc    :      1,920 ints
info_3x3   :      4,864 ints
info_convs :    410,400 ints
info_avg   :        760 ints
info_fc    :     12,160 ints


In [4]:
  from pynq import Overlay, MMIO, allocate
# 2) overlay + ip 확인
  ol = Overlay('mobilenet.bit')
  print("Bitstream loaded.")
  print("IPs:", list(ol.ip_dict.keys()))

  # CTRL_BUS 살아있는지 확인 (ap_idle = 1)
  ctrl = MMIO(0x40000000, 0x10000)
  ap = ctrl.read(0x00)
  print(f"\nap_ctrl reg = 0x{ap:08x}  →  ap_idle = {(ap>>2)&1}")


Bitstream loaded.
IPs: ['MobileNet_Stream_0', 'axi_gpio_0', 'axi_dma_0', 'axi_dma_1', 'axi_dma_2', 'processing_system7_0']

ap_ctrl reg = 0x00000004  →  ap_idle = 1


In [5]:
 # 3) DDR 버퍼 할당 + 데이터 복사
def alloc_and_copy(src):
    buf = allocate(shape=src.shape, dtype=np.int32)
    np.copyto(buf, src)
    buf.flush()
    return buf

t0 = time.time()
buf_w_conv     = alloc_and_copy(weights_CONV)
buf_w_fc       = alloc_and_copy(weights_FC)
buf_b_conv     = alloc_and_copy(bias_CONV)
buf_b_fc       = alloc_and_copy(bias_FC)
buf_tile_3x3   = alloc_and_copy(tile_3x3)
buf_tile_convs = alloc_and_copy(tile_convs)
buf_tile_avg   = alloc_and_copy(tile_avg)
buf_tile_fc    = alloc_and_copy(tile_fc)
buf_info_3x3   = alloc_and_copy(info_3x3)
buf_info_convs = alloc_and_copy(info_convs)
buf_info_avg   = alloc_and_copy(info_avg)
buf_info_fc    = alloc_and_copy(info_fc)
print(f"Allocated 12 buffers in {time.time()-t0:.1f}s")


Allocated 12 buffers in 0.1s


In [6]:
print('w_conv', hex(buf_w_conv.physical_address), buf_w_conv.nbytes)
print('w_fc', hex(buf_w_fc.physical_address), buf_w_fc.nbytes)
print('b_conv', hex(buf_b_conv.physical_address), buf_b_conv.nbytes)
print('b_fc', hex(buf_b_fc.physical_address), buf_b_fc.nbytes)
print('tile_3x3', hex(buf_tile_3x3.physical_address), buf_tile_3x3.nbytes)
print('tile_convs', hex(buf_tile_convs.physical_address),buf_tile_convs.nbytes)
print('tile_avg', hex(buf_tile_avg.physical_address), buf_tile_avg.nbytes)
print('tile_fc', hex(buf_tile_fc.physical_address), buf_tile_fc.nbytes)
print('info_3x3', hex(buf_info_3x3.physical_address), buf_info_3x3.nbytes)
print('info_convs', hex(buf_info_convs.physical_address),buf_info_convs.nbytes)
print('info_avg', hex(buf_info_avg.physical_address), buf_info_avg.nbytes)
print('info_fc', hex(buf_info_fc.physical_address), buf_info_fc.nbytes)


w_conv 0x15900000 6102624
w_fc 0x15f00000 1440000
b_conv 0x15860000 68224
b_fc 0x1584a000 4000
tile_3x3 0x1584c000 3072
tile_convs 0x15880000 259200
tile_avg 0x1584e000 480
tile_fc 0x15850000 7680
info_3x3 0x15858000 19456
info_convs 0x16100000 1641600
info_avg 0x15852000 3040
info_fc 0x158c0000 48640


In [7]:
# 4) 고정 포인터 4개 등록 (w_conv, b_conv, w_fc, b_fc)
# 4-1) 포인터 쓰기
data_mmio = MMIO(0x40010000, 0x10000)
data_mmio.write(0x10, buf_w_conv.physical_address & 0xFFFFFFFF)
data_mmio.write(0x14, (buf_w_conv.physical_address >> 32) & 0xFFFFFFFF)
data_mmio.write(0x1C, buf_b_conv.physical_address & 0xFFFFFFFF)
data_mmio.write(0x20, (buf_b_conv.physical_address >> 32) & 0xFFFFFFFF)
data_mmio.write(0x28, buf_w_fc.physical_address & 0xFFFFFFFF)
data_mmio.write(0x2C, (buf_w_fc.physical_address >> 32) & 0xFFFFFFFF)
data_mmio.write(0x34, buf_b_fc.physical_address & 0xFFFFFFFF)
data_mmio.write(0x38, (buf_b_fc.physical_address >> 32) & 0xFFFFFFFF)
print('4 pointers written.')

4 pointers written.


In [8]:
# 4-2) 검증 (read back)
print('w_conv lo', hex(data_mmio.read(0x10)), 'hi',
hex(data_mmio.read(0x14)))
print('b_conv lo', hex(data_mmio.read(0x1C)), 'hi',
hex(data_mmio.read(0x20)))
print('w_fc   lo', hex(data_mmio.read(0x28)), 'hi',
hex(data_mmio.read(0x2C)))
print('b_fc   lo', hex(data_mmio.read(0x34)), 'hi',
hex(data_mmio.read(0x38)))


w_conv lo 0x15900000 hi 0x0
b_conv lo 0x15860000 hi 0x0
w_fc   lo 0x15f00000 hi 0x0
b_fc   lo 0x1584a000 hi 0x0


In [9]:
# pynq에서 dma 객체 잡기
ip_in  = ol.axi_dma_0.sendchannel
ip_out = ol.axi_dma_1.recvchannel
res_rd = ol.axi_dma_2.sendchannel
res_wr = ol.axi_dma_2.recvchannel
print('DMA channels OK:', ip_in, ip_out, res_rd, res_wr)


DMA channels OK: <pynq.lib.dma._SDMAChannel object at 0xb385e1d8> <pynq.lib.dma._SDMAChannel object at 0xb385e688> <pynq.lib.dma._SDMAChannel object at 0xb385e0e8> <pynq.lib.dma._SDMAChannel object at 0xb385e2b0>


In [10]:
%run /home/xilinx/jupyter_notebooks/mobilenet/python/helpers.py

helpers.py loaded.
  ctrl @ 0x40000000, data @ 0x40010000
  ap_idle = 1


In [11]:
%run -i /home/xilinx/jupyter_notebooks/mobilenet/python/run_layer0.py

=== Packing input (this may take 30-60s in pure Python) ===
  packed input: 1,382,400 ints  (8.9s)
  buf_input phys = 0x16300000
  per-PE output buffer: 100,352 ints x 4 PE

=== Calling IP per PE ===
  ap_idle before = 1
  PE 0: tile=0x1584c000, info=0x15858000 ... done in 114.1 ms
  PE 1: tile=0x1584c300, info=0x15859300 ... done in 114.2 ms
  PE 2: tile=0x1584c600, info=0x1585a600 ... done in 114.2 ms
  PE 3: tile=0x1584c900, info=0x1585b900 ... done in 114.1 ms

=== Reassembling output ===
  PE 0: wrote 100,352 ints
  PE 1: wrote 100,352 ints
  PE 2: wrote 100,352 ints
  PE 3: wrote 100,352 ints

=== Output sanity ===
  shape: (401408,)
  min/max: 0 / 1229027
  nonzero ratio: 57.8%
  first 8 values: [0 0 0 0 0 0 0 0]


In [12]:
  print("Position [0:16]:    ", out_layer0[0:16])
  print("Position [56*112:56*112+16] (중앙 부근):", out_layer0[56*112:56*112+16])
  print("Channel 5 시작:     ", out_layer0[5*12544:5*12544+16])
  print("Channel 31 (마지막) 시작:", out_layer0[31*12544:31*12544+16])
  print("\n0이 아닌 위치 처음 5개:", np.nonzero(out_layer0)[0][:5])
  print("그 위치 값:", out_layer0[np.nonzero(out_layer0)[0][:5]])


Position [0:16]:     [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
Position [56*112:56*112+16] (중앙 부근): [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
Channel 5 시작:      [218414 231621 231831 231692 231692 231692 231456 231831 231588 231494
 231494 231662 231696 231696 231658 231522]
Channel 31 (마지막) 시작: [416921 409998 408503 408565 408565 408565 409501 408503 406487 406805
 406805 404554 404600 404600 403728 402792]

0이 아닌 위치 처음 5개: [12564 12565 12566 12567 12568]
그 위치 값: [231346 384846 340886 284111  10524]


In [13]:
  import numpy as np
  print("PE 0 raw buffer 첫 16:", np.asarray(buf_out_pe[0])[:16])
  print("PE 0 raw stats:        min=", buf_out_pe[0].min(),
        "max=", buf_out_pe[0].max(),
        "nonzero=", np.count_nonzero(buf_out_pe[0]),
        "/", len(buf_out_pe[0]))
  print("PE 0 raw nonzero 처음 위치:", np.nonzero(np.asarray(buf_out_pe[0]))[0][:5])
  print()
  print("PE 1 raw buffer 첫 16:", np.asarray(buf_out_pe[1])[:16])
  print("PE 1 raw stats:        min=", buf_out_pe[1].min(),
        "max=", buf_out_pe[1].max(),
        "nonzero=", np.count_nonzero(buf_out_pe[1]))
  print()
  print("Weight CONV 첫 16:    ", weights_CONV[:16])
  print("Weight CONV nonzero:  ", np.count_nonzero(weights_CONV[:1000]), "/1000 (앞 1000개 중)")


PE 0 raw buffer 첫 16: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
PE 0 raw stats:        min= 0 max= 1229027 nonzero= 72987 / 100352
PE 0 raw nonzero 처음 위치: [210 224 238 239 252]

PE 1 raw buffer 첫 16: [ 35953 319878 315877 316104 316104 316104 312301 315877 315406 315920
 315920 319680 315640 315640      0 325619]
PE 1 raw stats:        min= 0 max= 1216309 nonzero= 50577

Weight CONV 첫 16:     [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
Weight CONV nonzero:   780 /1000 (앞 1000개 중)


In [14]:
  print(buf_w_conv.physical_address, buf_b_conv.physical_address)
  print(buf_tile_convs.physical_address, buf_info_convs.physical_address)


361758720 361103360
361234432 370147328


In [15]:
 %run -i /home/xilinx/jupyter_notebooks/mobilenet/python/helpers_v2.py

helpers_v2.py loaded.
  Functions: dram_2_stream_3x3_type2, dram_2_stream_1x1_exp0/1,
             stream_2_dram_3x3_exp0/1, stream_2_dram_1x1_exp0,
             tile/info offset helpers, conv_batch_relu_pe


In [16]:
 %run -i /home/xilinx/jupyter_notebooks/mobilenet/python/run_layer1.py

=== Allocating cpu_map ===
  buf_cpu_map phys = 0x16a00000, 6,291,456 bytes
  cpu_map filled with layer 0 output (32x112x112 = 401408 ints)
  cpu_map[:8] = [0 0 0 0 0 0 0 0]

=== Layer 1 Group 1: depthwise 3x3 (in=32, out=32, k=3, s=1) ===
  PE 0: packed input 115,200 ints in 0.8s
    IP done in 3.5 ms
  PE 1: packed input 115,200 ints in 0.8s
    IP done in 3.5 ms
  PE 2: packed input 115,200 ints in 0.7s
    IP done in 3.4 ms
  PE 3: packed input 115,200 ints in 0.8s
    IP done in 3.6 ms

  Writing Group 1 output to cpu_map (sequential, expansion=0)
  Wrote 401,408 ints to cpu_map
  Group 1 stats: nonzero 69.1%, min=0, max=196608
  Group 1 first 16: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]

=== Layer 1 Group 2: 1x1 conv (in=32, out=16, k=1, s=1) ===
  PE 0: packed input 100,352 ints in 0.5s


TimeoutError: IP timeout, ap_ctrl=0x00000001

In [17]:
  import time

  # Layer 1 Group 1만 (이미 잘 됨) - 다시 채우기
  buf_l1g1_outputs = []
  for pe in range(NUMBER_PE):
      packed_in = dram_2_stream_3x3_type2(np.asarray(buf_cpu_map), 32, 112, pe)
      buf_in = allocate(packed_in.shape, dtype=np.int32); buf_in[:] = packed_in; buf_in.flush()
      buf_out = allocate((100352,), dtype=np.int32)
      conv_batch_relu_pe(1, 0, 2, pe, buf_in, buf_out, ip_in, ip_out, timeout=10)
      buf_l1g1_outputs.append(buf_out); del buf_in
  stream_2_dram_3x3_exp0(buf_cpu_map, buf_l1g1_outputs, 32, 112, 1)
  buf_cpu_map.flush()
  print("G1 done. ap_ctrl =", hex(ctrl.read(0)))
  print("cpu_map first 16:", np.asarray(buf_cpu_map)[:16])
  print("cpu_map[100352:100368] (PE 1 영역):", np.asarray(buf_cpu_map)[100352:100368])


RuntimeError: DMA channel not idle